In [36]:
import fitz
import pandas as pd

pdf = fitz.open("Stock Trader's Almanac 2026_L.pdf")

print("Pages:", len(pdf))

Pages: 273


Enter month

In [37]:
month = "SEPTEMBER"

Search file keywords

In [38]:
pages_found = []

for page_num in range(len(pdf)):

    text = pdf[page_num].get_text()

    if f"{month} ALMANAC" in text:

        pages_found.append(page_num + 1)

print("Found pages:", pages_found)

Found pages: [3, 116]


Select real page

In [39]:
real_page = max(pages_found)

print("Real Almanac Page:", real_page)

Real Almanac Page: 116


Show the content

In [40]:
page = pdf[real_page - 1]

text = page.get_text()

print(text[:5000])

SEPTEMBER ALMANAC
Market Probability Chart above is a graphic representation of the S&P 500 Recent
Market Probability Calendar on page 126.
◆ Portfolio managers back after Labor Day tend to clean house in September ◆
Biggest % loser on the S&P, Dow and NASDAQ since 1950 (pages 52 & 60) ◆ Streak
of four great Dow Septembers averaging 4.2% gains ended in 1999 with six losers in a
row averaging –5.9% (page 156), up four straight 2005–2007, down 6% in 2008 and
2011, up 7.7% in 2010, down big in four of last five years ◆ Day after Labor Day Dow
down 13 of last 16 ◆ S&P opened strong 18 of last 30 years but tends to close weak
due to end-of-quarter mutual fund portfolio restructuring, last trading day: S&P
down 18 of past 28 ◆ September Triple-Witching Week can be dangerous; week after
is pitiful (page 108)
September Vital Statistics
DJIA
S&P 500
NASDAQ
Russell 1K Russell 2K
Rank
12
12
12
12
12
Up
30
33
28
22
24
Down
45
41
26
24
22
Average %
Change
–0.8
–0.7
–0.9
–0.9
–0.8
Midterm Yr Avg
% C

Preserve evidence

In [41]:
pix = page.get_pixmap(
    matrix=fitz.Matrix(4,4)
)

image_name = f"{month.lower()}_vital_statistics.png"

pix.save(image_name)

print(image_name)

september_vital_statistics.png


In [47]:
import re

def find_line(keyword):

    for i, line in enumerate(lines):

        if line.strip().lower() == keyword.lower():
            return i

    raise ValueError(f"{keyword} not found")


rank_idx = find_line("Rank")
up_idx = find_line("Up")
down_idx = find_line("Down")
avg_idx = next(i for i, x in enumerate(lines) if x.startswith("Average"))
mid_idx = next(i for i, x in enumerate(lines) if x.startswith("Midterm"))
best_idx = next(i for i, x in enumerate(lines) if x.startswith("Best & Worst"))


def read_values(start, end):

    nums = []

    for line in lines[start+1:end]:

        nums.extend(
            re.findall(r'-?\d+\.?\d*', line.replace("–","-"))
        )

    return nums


rank_row = read_values(rank_idx, up_idx)
up_row = read_values(up_idx, down_idx)
down_row = read_values(down_idx, avg_idx)
avg_row = read_values(avg_idx, mid_idx)
mid_row = read_values(mid_idx, best_idx)

print(rank_row)
print(up_row)
print(down_row)
print(avg_row)
print(mid_row)

rank = int(rank_row[1])
up_years = int(up_row[1])
down_years = int(down_row[1])
avg_return = float(avg_row[1])
midterm_return = float(mid_row[1])

print()
print("Rank:", rank)
print("Up Years:", up_years)
print("Down Years:", down_years)
print("Average Return:", avg_return)+
print("Midterm Return:", midterm_return)

['12', '12', '12', '12', '12']
['30', '33', '28', '22', '24']
['45', '41', '26', '24', '22']
['-0.8', '-0.7', '-0.9', '-0.9', '-0.8']
['-1.2', '-0.8', '-1.6', '-1.8', '-1.6']

Rank: 12
Up Years: 33
Down Years: 41
Average Return: -0.7
Midterm Return: -0.8


Almanac Data Table

In [48]:
stats_df = pd.DataFrame({
    "Metric": [
        "Rank",
        "Up Years",
        "Down Years",
        "Average Return (%)",
        "Midterm Return (%)"
    ],
    "Value": [
        rank,
        up_years,
        down_years,
        avg_return,
        midterm_return
    ]
})

stats_df

,Metric,Value
0,Rank,12.0
1,Up Years,33.0
2,Down Years,41.0
3,Average Return (%),-0.7
4,Midterm Return (%),-0.8


Verdict is automatically generated based on Almanac data.

In [49]:
if midterm_return >= 1:
    verdict = "Bullish"

elif midterm_return > 0:
    verdict = "Slightly Bullish"

elif midterm_return > -1:
    verdict = "Slightly Bearish"

else:
    verdict = "Bearish"

print("Verdict:", verdict)

Verdict: Slightly Bearish


Evidence Table

In [50]:
evidence_df = pd.DataFrame({
    "Evidence": [
        f"{month.title()} Average Return {avg_return}%",
        f"{month.title()} Midterm Return {midterm_return}%"
    ],
    "Impact": [
        "Bullish" if avg_return > 0 else "Bearish",
        "Bullish" if midterm_return > 0 else "Bearish"
    ]
})

evidence_df

,Evidence,Impact
0,September Average Return -0.7%,Bearish
1,September Midterm Return -0.8%,Bearish


Export Evidence CSV file

In [51]:
evidence_df.to_csv(
    f"{month.lower()}_almanac_evidence.csv",
    index=False
)

print(
    f"{month.lower()}_almanac_evidence.csv created"
)

september_almanac_evidence.csv created


Generate Almanac Markdown report

In [52]:
md_text = f"""
# Almanac Agent Output

## Month

{month.title()} 2026

## Statistics

Rank: {rank}

Up Years: {up_years}

Down Years: {down_years}

Average Return: {avg_return}%

Midterm Return: {midterm_return}%

## Final Verdict

{verdict}

## Evidence

- {month.title()} Average Return {avg_return}%
- {month.title()} Midterm Return {midterm_return}%
"""

with open(
    f"{month.lower()}_almanac.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(md_text)

print(
    f"{month.lower()}_almanac.md created"
)

september_almanac.md created
